<a href="https://colab.research.google.com/github/krithi-ks/flyrankAI-ML/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krithi-ks/flyrankAI-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

### Rule

Prioritize a content item when it has enough search exposure to justify review and its observed CTR is below the typical CTR for content in a similar search-position range.

The score combines **search opportunity** and **CTR gap**. Higher impressions increase the potential opportunity, while a larger negative CTR gap increases the review priority.

### Reason code

**`ctr_opportunity`** — the page has sufficient search exposure and its CTR is below the observed position-bucket benchmark.

### Action label

**`Review title/snippet`** — review the page's search-facing title and snippet for a possible CTR improvement.

This is a transparent rule-based baseline. It uses only March 2026 observed signals and does not use future windows or label-derived fields.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ML-07 Section 2 — Build the ranked queue

import os
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# Step 1: Aggregate March 2026 data to client × content
# ---------------------------------------------------------

baseline_df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,

        AVG(
            CASE
                WHEN gsc_avg_position > 0
                THEN gsc_avg_position
                ELSE NULL
            END
        ) AS avg_position

    FROM {PERFORMANCE}

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("Client × content rows:", len(baseline_df))


# ---------------------------------------------------------
# Step 2: Calculate observed CTR
# ---------------------------------------------------------

baseline_df["ctr_pct"] = (
    100.0
    * baseline_df["clicks"]
    / baseline_df["impressions"].replace(0, np.nan)
)


# ---------------------------------------------------------
# Step 3: Create position buckets
# ---------------------------------------------------------

def position_bucket(position):
    if pd.isna(position):
        return "no_position"
    elif position <= 3:
        return "1-3"
    elif position <= 10:
        return "4-10"
    elif position <= 20:
        return "11-20"
    else:
        return "21+"

baseline_df["position_bucket"] = (
    baseline_df["avg_position"]
    .apply(position_bucket)
)


# ---------------------------------------------------------
# Step 4: Calculate position-bucket CTR benchmark
# ---------------------------------------------------------

position_benchmark = (
    baseline_df[
        (baseline_df["impressions"] > 0)
        & (baseline_df["ctr_pct"].notna())
        & (baseline_df["position_bucket"] != "no_position")
    ]
    .groupby("position_bucket")["ctr_pct"]
    .median()
    .to_dict()
)

print("Position-bucket CTR benchmarks:")
print(position_benchmark)


# ---------------------------------------------------------
# Step 5: Add benchmark and CTR gap
# ---------------------------------------------------------

baseline_df["benchmark_ctr_pct"] = (
    baseline_df["position_bucket"]
    .map(position_benchmark)
)

baseline_df["ctr_gap_pct"] = (
    baseline_df["benchmark_ctr_pct"]
    - baseline_df["ctr_pct"]
)


# ---------------------------------------------------------
# Step 6: Define the transparent review rule
# ---------------------------------------------------------

baseline_df["eligible"] = (
    (baseline_df["impressions"] >= 500)
    & (baseline_df["avg_position"].notna())
    & (baseline_df["ctr_pct"].notna())
    & (baseline_df["benchmark_ctr_pct"].notna())
    & (baseline_df["ctr_gap_pct"] > 0)
)


# ---------------------------------------------------------
# Step 7: Score
#
# Score = impressions × relative CTR gap
#
# This is deliberately NOT a fitted model.
# ---------------------------------------------------------

baseline_df["relative_ctr_gap"] = (
    baseline_df["ctr_gap_pct"]
    / baseline_df["benchmark_ctr_pct"].replace(0, np.nan)
)

baseline_df["score"] = np.where(
    baseline_df["eligible"],
    baseline_df["impressions"] * baseline_df["relative_ctr_gap"],
    0.0
)


# ---------------------------------------------------------
# Step 8: Reason code and action
# ---------------------------------------------------------

baseline_df["reason_code"] = np.where(
    baseline_df["eligible"],
    "ctr_opportunity",
    "not_prioritized"
)

baseline_df["action"] = np.where(
    baseline_df["eligible"],
    "Review title/snippet",
    "No immediate action"
)


# ---------------------------------------------------------
# Step 9: Rank
# ---------------------------------------------------------

baseline_df = baseline_df.sort_values(
    ["score", "impressions"],
    ascending=[False, False]
).reset_index(drop=True)

baseline_df["rank"] = np.arange(1, len(baseline_df) + 1)


# ---------------------------------------------------------
# Step 10: Save required output
# ---------------------------------------------------------

output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)

queue_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "impressions",
    "clicks",
    "avg_position",
    "ctr_pct",
    "benchmark_ctr_pct",
    "ctr_gap_pct",
    "score",
    "reason_code",
    "action"
]

baseline_queue = baseline_df[queue_columns].copy()

output_path = os.path.join(
    output_dir,
    "baseline_action_score.csv"
)

baseline_queue.to_csv(
    output_path,
    index=False
)

print(f"Saved ranked queue to: {output_path}")
print(f"Total rows: {len(baseline_queue):,}")
print(f"Prioritized rows: {(baseline_queue['score'] > 0).sum():,}")

display(baseline_queue.head(10))

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*


I review the top 20 ranked items as a human rather than treating the score as ground truth. The action is to review the title/snippet because the rule identifies a CTR opportunity relative to the observed position benchmark.

The confidence is higher when both search exposure and the CTR gap are substantial. The recommendation could be wrong if the observed CTR difference is caused by query mix, seasonality, SERP features, measurement differences, or another factor not represented in the baseline.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ML-07 Section 3 — Top-20 review

top20 = baseline_queue[
    baseline_queue["score"] > 0
].head(20).copy()

top20["confidence_note"] = np.where(
    (
        (top20["impressions"] >= 1000)
        & (top20["ctr_gap_pct"] >= 1.0)
    ),
    "Higher confidence: strong exposure and CTR gap",
    "Lower confidence: signal is weaker or less exposed"
)

top20["what_would_make_it_wrong"] = (
    "The CTR gap may reflect query mix, seasonality, "
    "SERP features, or measurement differences."
)

review_columns = [
    "rank",
    "action",
    "reason_code",
    "confidence_note",
    "what_would_make_it_wrong"
]

display(top20[review_columns])

NameError: name 'baseline_queue' is not defined

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


The baseline is intentionally simple, so some high-ranked items can be weak picks. A page can have a large CTR gap because of query mix, SERP features, seasonality, or measurement effects rather than because its title or snippet needs improvement.

I therefore treat the ranked queue as a decision-support list rather than a confirmed diagnosis. The rule uses only March 2026 observed performance signals. No future-period measurements, label-derived fields, product flags, client names, or private information are used.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ML-07 Section 4 — Weak picks + leakage check

# ---------------------------------------------------------
# Weak-pick check
# ---------------------------------------------------------

weak_picks = baseline_queue[
    (baseline_queue["score"] > 0)
    & (
        (baseline_queue["impressions"] < 1000)
        | (baseline_queue["ctr_gap_pct"] < 1.0)
    )
].head(5)

print("Potential weak picks:")
display(
    weak_picks[
        [
            "rank",
            "impressions",
            "ctr_pct",
            "benchmark_ctr_pct",
            "ctr_gap_pct",
            "score",
            "reason_code",
            "action"
        ]
    ]
)


# ---------------------------------------------------------
# Leakage check
# ---------------------------------------------------------

feature_columns = [
    "impressions",
    "clicks",
    "avg_position",
    "ctr_pct",
    "benchmark_ctr_pct",
    "ctr_gap_pct"
]

forbidden_terms = [
    "trend",
    "declining",
    "label",
    "future",
    "next",
    "product_flag"
]

leaky_features = [
    col for col in feature_columns
    if any(term in col.lower() for term in forbidden_terms)
]

print("\nFeatures used by baseline:")
print(feature_columns)

print("\nPotential leakage-named features:")
print(leaky_features)

if len(leaky_features) == 0:
    print("\nPASS: No future-window or label-derived feature names are used.")
else:
    print("\nCHECK REQUIRED: Potential leakage detected.")

NameError: name 'baseline_queue' is not defined

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.